In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import polars as pl
import seaborn as sns
from keplergl import KeplerGl
colores_datahub={
        "rojo":"#c84950",
        "morado": "#88418c",
        "celeste":"#5385c3",
        "turquesa":"#50adbc",
        "azul":"#265282"
         }

# Filtrar solo síndrome metabólico y columnas necesarias para flujos
DATE_FORMAT = "%Y-%m-%d"

In [2]:
cantones = gpd.read_file("../../data/organizacion-territorial-cantonal/ORGANIZACION_TERRITORIAL_CANTONAL.shp")

# Ver el CRS actual
print(cantones.crs)
# Convertir a lat/lon (EPSG:4326)
cantones = cantones.to_crs(epsg=4326)
cantones["centroid"] = cantones.geometry.centroid
cantones["lat_canton"] = cantones.centroid.y
cantones["lon_canton"] = cantones.centroid.x
cantones["DPA_CANTON"] = "EC" + cantones["DPA_CANTON"].astype(str)
cantones.rename(columns={"DPA_CANTON":"code_canton"},inplace=True)

EPSG:32717


C:\Users\Kristian Mendoza\AppData\Local\Temp\ipykernel_21812\575717680.py:7: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cantones["centroid"] = cantones.geometry.centroid
C:\Users\Kristian Mendoza\AppData\Local\Temp\ipykernel_21812\575717680.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cantones["lat_canton"] = cantones.centroid.y
C:\Users\Kristian Mendoza\AppData\Local\Temp\ipykernel_21812\575717680.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cantones["lon_canton"] = cantones.centroid.x


In [3]:
complete_path = "../../data/data_hospitales/sistema_salud_egresos_limpio.parquet"

# 3. Write the Polars DataFrame to a Parquet file
base_general= pl.read_parquet(complete_path)

In [ ]:
print(base_general.columns)

In [4]:
identificadores = [
    'clase',
    'tipo',
    'entidad',
    'sector'
]
territorio_ubi=[    
    'prov_ubi',
    'cant_ubi',
    'parr_ubi',
    #'area_ubi',

]
territorio_res = [
    'prov_res',
    'cant_res',
    'parr_res',
    #'area_res',
]

fecha_egr = [
    'fecha_egr',

]

estadisticas = [
    'dia_estad', #dias de estadia
    'con_egrpa', # egreso vivo o muerto (antes o despues de 48 horas)
]

diagnostico=[
    "cau_cie10",
    "causa3",
    "cap221rx",
    "cau221rx",
    "cau298rx",
]

demografia=[
    "sector",
    "mes_inv",
    "nac_pac",
    "nom_pais",
    "cod_pais",
    "sexo",
    "cod_edad",
    "edad",
    "etnia",
    
]


## Perfil epideiologico 

- Mapa del ecuador. nivel cantonal. mayor concentracin de capitulos en cantones de residencia. 
	- Filtrar por carga de enfermedad a nivel de capitulos.
	- Mostrar un filtro de solo sindrome metabolico.
	- Flujos filtrado por capitulo todos los años.

In [ ]:
perfil_epdiemiologico = (
    base_general
    .select([
        "code_cant_res","cant_res",
        "cau221rx_std","sindrome_metabolico",
        "fecha_egr",
        "anio_egr",
    ])
    .with_columns(
        pl.col("fecha_egr")
          .str.to_date(DATE_FORMAT)
          .dt.truncate("1y")
          .alias("fecha_egr_mes")
    )
    .drop_nulls(subset=["fecha_egr_mes"])
    .group_by([
        "code_cant_res","cant_res",
        "cau221rx_std","sindrome_metabolico",
        "anio_egr",
    ])
    .agg([
        pl.len().alias("conteo"),
    ])
    .to_pandas()
)

print(f"Registros síndrome metabólico: {perfil_epdiemiologico.shape[0]:,}")
print(perfil_epdiemiologico.head(3))


In [ ]:

perfil_epdiemiologico=perfil_epdiemiologico.merge(
    cantones[["code_canton","lat_canton","lon_canton"]],
    left_on="code_cant_res",
    right_on="code_canton",
    how="left"
).drop(columns=["code_canton"])


In [ ]:
perfil_epdiemiologico_sindrome = (
    base_general
    .filter(pl.col("sindrome_metabolico") == True)
    .select([
        "code_cant_res","cant_res",
        "cau221rx_std","sindrome_metabolico",
        "fecha_egr",
        "anio_egr",
    ])
    .with_columns(
        pl.col("fecha_egr")
          .str.to_date(DATE_FORMAT)
          .dt.truncate("1y")
          .alias("fecha_egr_mes")
    )
    .drop_nulls(subset=["fecha_egr_mes"])
    .group_by([
        "code_cant_res","cant_res",
        "cau221rx_std","sindrome_metabolico",
        "anio_egr",
    ])
    .agg([
        pl.len().alias("conteo"),
    ])
    .to_pandas()
)

perfil_epdiemiologico_sindrome=perfil_epdiemiologico_sindrome.merge(
    cantones[["code_canton","lat_canton","lon_canton"]],
    left_on="code_cant_res",
    right_on="code_canton",
    how="left"
).drop(columns=["code_canton"])


In [ ]:
print(f"Registros síndrome metabólico: {perfil_epdiemiologico.shape[0]:,}")
print(perfil_epdiemiologico.head(3))
print(f"Registros síndrome metabólico sindrome: {perfil_epdiemiologico_sindrome.shape[0]:,}")
print(perfil_epdiemiologico_sindrome.head(3))


In [ ]:
mapa = KeplerGl(height=800)
mapa.add_data(data=perfil_epdiemiologico, name="perfil_epdiemiologico")
mapa.add_data(data=perfil_epdiemiologico_sindrome, name="perfil_epdiemiologico_sindrome")
mapa

In [ ]:
# Opcional: exportar el mapa a HTML
with open('hex_config_perfil_epidemiologico.py', 'w') as f:
   f.write('config_perfil_epidemiologico = {}'.format(mapa.config))


In [ ]:
mapa.save_to_html(file_name="../../maps/for_presentation/processed/perfil_epidemiologico.html")

In [8]:
top_10_cau221rx_std=(
    base_general
    .select([
        "cau221rx_std",
    ])
    .group_by("cau221rx_std")
    .agg(pl.len().alias("conteo"))
    .sort("conteo", descending=True)
    .head(10)
)
top_10_cau221rx_std

cau221rx_std,conteo
str,u32
"""Parto (O80-O84)""",1313983
"""Trastornos de la vesícula bili…",575752
"""Atención materna relacionada c…",433325
"""Enfermedades del apéndice (K35…",416394
"""Complicaciones del trabajo de …",412566
"""Influenza [gripe] y neumonía (…",374519
"""Enfermedades infecciosas intes…",264353
"""Hernia (K40-K46)""",244247
"""Embarazo terminado en aborto (…",240955


In [12]:
flujos = (
    base_general
    .filter((pl.col("sindrome_metabolico") == True )| (pl.col("cau221rx_std").is_in(top_10_cau221rx_std["cau221rx_std"])))
    .select([
        "entidad",
        "code_cant_res","cant_res",
        "code_cant_ubi","cant_ubi",
        "cau221rx_std",
        "fecha_egr",
        "anio_egr","sindrome_metabolico",
    ])
    .with_columns(
        pl.col("fecha_egr")
          .str.to_date(DATE_FORMAT)
          .dt.truncate("1y")
          .alias("fecha_egr_anio")
    )
    .drop_nulls(subset=["fecha_egr_anio"])
    .group_by([
        "entidad",
        "code_cant_res","cant_res",
        "code_cant_ubi","cant_ubi",
        "cau221rx_std",
        "fecha_egr_anio","sindrome_metabolico"
    ])
    .agg([
        pl.len().alias("conteo"),
    ])
    .to_pandas()
)
#Conversión a timestamp — mismo enfoque que funcionaba antes
flujos["fecha_egr_anio"] = flujos["fecha_egr_anio"].dt.to_period("Y").dt.to_timestamp()

print(f"Flujos únicos: {len(flujos):,}")
print(f"Tipo fecha_egr_anio: {flujos['fecha_egr_anio'].dtype}")
print(f"Años disponibles: {sorted(flujos['fecha_egr_anio'].unique())}")

# Convertir a milisegundos (Unix Timestamp)
# Convertir a string con formato ISO completo
flujos["fecha_egr_anio"] = flujos["fecha_egr_anio"].dt.strftime('%Y-%m-%dT%H:%M:%S')

print(f"Tipo: {flujos['fecha_egr_anio'].dtype}") # Saldrá 'object'
print(flujos["fecha_egr_anio"].iloc[0]) # Verás '2015-04-01T00:00:00'

flujos=flujos.merge(
    cantones[["code_canton","lat_canton","lon_canton"]],
    left_on="code_cant_res",
    right_on="code_canton",
    how="left"
).drop(columns=["code_canton"]).rename(columns={"lat_canton":"lat_res","lon_canton":"lon_res"})

flujos=flujos.merge(
    cantones[["code_canton","lat_canton","lon_canton"]],
    left_on="code_cant_ubi",
    right_on="code_canton",
    how="left"
).drop(columns=["code_canton"]).rename(columns={"lat_canton":"lat_ubi","lon_canton":"lon_ubi"})


# quedarnos con el top 10 de cau221rx_std 
flujos_top10 = flujos.groupby("cau221rx_std")["conteo"].sum().sort_values(ascending=False).head(10).index


flujos_para_mapa = flujos[flujos["cau221rx_std"].isin(flujos_top10) | (flujos["sindrome_metabolico"] == True)]



print(f"Registros para visualizacion: {flujos_para_mapa.shape[0]:,}")
print(flujos_para_mapa.head(3))

C:\Users\Kristian Mendoza\AppData\Local\Temp\ipykernel_21812\3564825245.py:3: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .filter((pl.col("sindrome_metabolico") == True )| (pl.col("cau221rx_std").is_in(top_10_cau221rx_std["cau221rx_std"])))


Flujos únicos: 221,987
Tipo fecha_egr_anio: datetime64[ns]
Años disponibles: [Timestamp('2014-01-01 00:00:00'), Timestamp('2015-01-01 00:00:00'), Timestamp('2016-01-01 00:00:00'), Timestamp('2017-01-01 00:00:00'), Timestamp('2018-01-01 00:00:00'), Timestamp('2019-01-01 00:00:00'), Timestamp('2020-01-01 00:00:00'), Timestamp('2021-01-01 00:00:00'), Timestamp('2022-01-01 00:00:00'), Timestamp('2023-01-01 00:00:00'), Timestamp('2024-01-01 00:00:00')]
Tipo: object
2015-01-01T00:00:00
Registros para visualizacion: 221,987
                       entidad code_cant_res   cant_res code_cant_ubi  \
0  MINISTERIO DE SALUD PUBLICA        EC1101       LOJA        EC1902   
1  PRIVADOS CON FINES DE LUCRO        EC1308      MANTA        EC1308   
2  PRIVADOS CON FINES DE LUCRO        EC1705  RUMINAHUI        EC1705   

    cant_ubi                                       cau221rx_std  \
0  CHINCHIPE                                   Hernia (K40-K46)   
1      MANTA             Embarazo terminado en abo

In [13]:
flujos_para_mapa["cau221rx_std"].value_counts()

cau221rx_std
Parto (O80-O84)                                                                                                26893
Trastornos de la vesícula biliar, de vías biliares y páncreas (K80-K87)                                        26106
Enfermedades del apéndice (K35-K38)                                                                            20253
Influenza [gripe] y neumonía (J09-J18)                                                                         19234
Hernia (K40-K46)                                                                                               18723
Atención materna relacionada con el feto y la cavidad amniótica y con posibles problemas del parto(O30-O48)    16665
Otros trastornos maternos relacionados principalmente con el  embarazo (O20-O29)                               15229
Complicaciones del trabajo de parto y del parto (O60-O75)                                                      15215
Diabetes mellitus (E10-E14)                        

In [14]:
%run hex_config_flujos.py


In [15]:
mapa_flujos = KeplerGl(height=800, config=config_flujos)
mapa_flujos.add_data(data=flujos_para_mapa, name="flujos")
#mapa_flujos.add_data(data=cantones_ubi, name="cantones_ubi")
mapa_flujos


User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


c:\Users\Kristian Mendoza\.conda\envs\rodri_msp_geo\Lib\site-packages\jupyter_client\session.py:721: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)


KeplerGl(config={'version': 'v1', 'config': {'visState': {'filters': [{'dataId': ['flujos'], 'id': 'avymmdin6'…

In [16]:
# Opcional: exportar el mapa a HTML
with open('hex_config_flujos.py', 'w') as f:
   f.write('config_flujos = {}'.format(mapa_flujos.config))

In [ ]:
mapa_flujos.save_to_html(file_name="../../maps/for_presentation/processed/flujos.html")

Map saved to ../../maps/processed/flujos.html!


## Produccion hospitalaria:
TABLAS TODO 
- A nivel nacional: estadisticas generales por tipo de hospital. Clase puede ser. ultimo año nota que puede ser. interactivo
-  TABLA
	- cantidad de egresos hospitalarios. (epsecialidad, general,etc)
- Dias promedios de esta por tipo de hospital ( publico privado, iess, no lucro). Ejemplo: capitulos del sindrome metabolico.
- 

In [73]:
base_general.select([*identificadores]).unique()

clase,tipo,entidad,sector
str,str,str,str
"""ESTABLECIMIENTOS DEL DIA QUE P…","""Agudo""","""PRIVADOS CON FINES DE LUCRO""","""Privado con fines de lucro"""
"""HOSPITAL DEL DIA CON INTERNACI…","""Agudo""","""PRIVADOS SIN FINES DE LUCRO""","""Privado sin fines de lucro"""
"""ESTABLECIMIENTOS DEL DIA QUE P…","""Agudo""","""MINISTERIO DE SALUD PUBLICA""","""Público"""
"""DERMATOLOGICO (LEPROCOMIOS)""","""Crónico""","""MINISTERIO DE SALUD PUBLICA""","""Público"""
"""HOSPITAL DEL DIA CON INTERNACI…","""Agudo""","""MUNICIPIOS""","""Público"""
…,…,…,…
"""CLINICA GENERAL (SIN ESPECIALI…","""Clínicas generales sin especia…","""PRIVADOS SIN FINES DE LUCRO""","""Privado sin fines de lucro"""
"""PSIQUIATRICA""","""Crónico""","""PRIVADOS CON FINES DE LUCRO""","""Privado con fines de lucro"""
"""NEUMOLOGICO (ANTITUBERCULOSO)""","""Crónico""","""MINISTERIO DE SALUD PUBLICA""","""Público"""


In [76]:
base_general.select(["clase"]).unique()

clase
str
"""PSIQUIATRICO Y SANATORIO DE AL…"
"""HOSPITAL GENERAL"""
"""PEDIATRICO"""
"""GINECO-OBSTETRICA"""
"""NEUMOLOGICO (ANTITUBERCULOSO)"""
…
"""OTRAS CLINICAS ESPECIALIZADAS"""
"""CLINICA GENERAL (SIN ESPECIALI…"
"""HOSPITAL DEL DIA CON INTERNACI…"


In [18]:
# con clase saquemos estadisticas generales: estadisticas = [
#    'dia_estad', #dias de estadia
#    'con_egrpa', # egreso vivo o muerto (antes o despues de 48 horas)
    #
produccion_hospitalaria_conteo_por_clase=(base_general.filter(pl.col("anio_egr")!="2014.0").select([*identificadores,*estadisticas,"anio_egr"]).group_by("clase","anio_egr").agg([
        pl.len().alias("conteo"),
    ]).to_pandas())

produccion_hospitalaria_conteo_por_entidad=(base_general.filter(pl.col("anio_egr")!="2014.0").select([*identificadores,*estadisticas,"anio_egr"]).group_by("entidad","anio_egr").agg([
        pl.len().alias("conteo"),
    ]).to_pandas())

print(produccion_hospitalaria_conteo_por_clase.sort_values("anio_egr", ascending=True).head(10))


                                 clase anio_egr  conteo
0                           ONCOLOGICO   2015.0   26690
55                    HOSPITAL GENERAL   2015.0  446587
32                   GINECO-OBSTETRICA   2015.0    3545
71  CLINICA GENERAL (SIN ESPECIALIDAD)   2015.0  253633
78                          GERIATRICO   2015.0     951
79                        INFECTOLOGIA   2015.0    1568
22                   GINECO-OBSTETRICO   2015.0   79085
47         DERMATOLOGICO (LEPROCOMIOS)   2015.0     419
20                       TRAUMATOLOGIA   2015.0     982
21          HOSPITAL DE ESPECIALIDADES   2015.0   95569


In [ ]:
# import plotly.express as px
# import plotly.graph_objects as go
# import pandas as pd

# # 1. Preparar el gráfico de evolución temporal
# fig_lineas = px.line(
#     produccion_hospitalaria_conteo_por_clase.sort_values("anio_egr"), 
#     x="anio_egr", 
#     y="conteo", 
#     color="clase",
#     title="Evolución Temporal de Altas Hospitalarias por Clase",
#     markers=True,
#     labels={"anio_egr": "Año", "conteo": "Número de Altas", "clase": "Clase del Establecimiento"},
#     template="plotly_white"
# )
# fig_lineas = px.line(
#     produccion_hospitalaria_conteo_por_entidad.sort_values("anio_egr"), 
#     x="anio_egr", 
#     y="conteo", 
#     color="entidad",
#     title="Evolución Temporal de Altas Hospitalarias por Entidad",
#     markers=True,
#     labels={"anio_egr": "Año", "conteo": "Número de Altas", "entidad": "Entidad del Establecimiento"},
#     template="plotly_white"
# )
# # 2. Crear la Tabla Dinámica (con filtros de año)
# # Usamos un objeto Table de Plotly que permite ver los datos, 
# # aunque para filtrado avanzado "tipo Excel" en un HTML estático 
# # lo ideal es exportar el DataFrame procesado.



# # 3. Exportar a HTML (Ambos elementos en un solo archivo)
# with open("reporte_hospitalario.html", "w") as f:
#     f.write(fig_lineas.to_html(full_html=False, include_plotlyjs='cdn'))
#     f.write(fig_lineas.to_html(full_html=False, include_plotlyjs='cdn'))

# print("Reporte generado: reporte_hospitalario.html")

Reporte generado: reporte_hospitalario.html


In [24]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

# 1. Crear las "trazas" base para Clase y Entidad
# Usamos px.line pero solo para extraer los datos de las líneas
fig_clase = px.line(
    produccion_hospitalaria_conteo_por_clase.sort_values("anio_egr"), 
    x="anio_egr", y="conteo", color="clase", markers=True
)

fig_entidad = px.line(
    produccion_hospitalaria_conteo_por_entidad.sort_values("anio_egr"), 
    x="anio_egr", y="conteo", color="entidad", markers=True
)

# 2. Crear el objeto Figura final vacío
fig = go.Figure()

# Añadir las líneas de CLASE (Grupo 1)
for trace in fig_clase.data:
    trace.visible = "legendonly"  # <-- Esto las desactiva al inicio
    trace.name = f"Clase: {trace.name}" # Para diferenciar en la leyenda
    fig.add_trace(trace)

# Añadir las líneas de ENTIDAD (Grupo 2)
for trace in fig_entidad.data:
    trace.visible = False  # Al inicio están totalmente ocultas
    trace.name = f"Entidad: {trace.name}"
    fig.add_trace(trace)

# 3. Configurar los botones de alternancia
# Necesitamos saber cuántas trazas hay de cada una
num_clase = len(fig_clase.data)
num_entidad = len(fig_entidad.data)

fig.update_layout(
    updatemenus=[
        dict(
            type="buttons",
            direction="left",
            buttons=list([
                dict(
                    label="Ver por Clase",
                    method="update",
                    # Visibilidad: Clase como 'legendonly', Entidad como False
                    args=[{"visible": ["legendonly"] * num_clase + [False] * num_entidad},
                          {"title": "Altas Hospitalarias por Clase"}]
                ),
                dict(
                    label="Ver por Entidad",
                    method="update",
                    # Visibilidad: Clase como False, Entidad como 'legendonly'
                    args=[{"visible": [False] * num_clase + ["legendonly"] * num_entidad},
                          {"title": "Altas Hospitalarias por Entidad"}]
                ),
            ]),
            pad={"r": 12, "t": 12},
            showactive=True,
            x=0.5,
            xanchor="center",
            y=1.5,
            yanchor="top"
        ),
    ],
    template="plotly_white",
    title="Evolución Temporal Hospitalaria (Seleccione Categoría)",
    xaxis_title="Año de Egreso",
    yaxis_title="Número de Altas",
    legend_title="Leyenda (Clic para activar línea)"
)

# 4. Exportar a HTML
fig.write_html("reporte_hospitalario_interactivo.html", include_plotlyjs='cdn')

print("Reporte interactivo generado: reporte_hospitalario_interactivo.html")

Reporte interactivo generado: reporte_hospitalario_interactivo.html


In [108]:
# Agrupamos por entidad y año de egreso
dias_promedio_entidad = (
    base_general.filter(pl.col("anio_egr")!="2014.0")
    .with_columns(
        pl.col("dia_estad").cast(pl.Float64, strict=False),
        pl.col("anio_egr").cast(pl.Float64, strict=False)
    )
    .group_by(["entidad", "anio_egr"])
    .agg([
        pl.col("dia_estad").mean().alias("promedio_estadia")
    ])
    .to_pandas()
)

# Limpieza de años para evitar el error anterior
dias_promedio_entidad["anio_egr"] = pd.to_numeric(dias_promedio_entidad["anio_egr"]).astype(int)

# Creamos el pivot
tabla_estadia = dias_promedio_entidad.pivot_table(
    index="entidad",
    columns="anio_egr",
    values="promedio_estadia",
    aggfunc="first", # Ya es un promedio, así que 'first' o 'mean' funcionan igual
    fill_value=0
)

# Opcional: Redondear a 2 decimales para que no se vea feo el HTML
tabla_estadia = tabla_estadia.round(2)

# Limpiar nombres de ejes
tabla_estadia.columns.name = None
tabla_estadia.index.name = "Entidad Hospitalaria"
tabla_estadia = tabla_estadia.reset_index()

print(tabla_estadia.head())

                                Entidad Hospitalaria  2015  2016  2017  2018  \
0                                    FISCOMISIONALES  0.00  0.00  0.00  2.43   
1          INSTITUTO ECUATORIANO DE SEGURIDAD SOCIAL  4.96  4.70  4.66  4.75   
2                    JUNTA BENEFICENCIA DE GUAYAQUIL  5.86  6.33  7.79  8.77   
3                     MINISTERIO DE DEFENSA NACIONAL  4.89  4.69  4.75  4.76   
4  MINISTERIO DE JUSTICIA, DERECHOS HUMANOS Y CULTOS  5.41  5.43  5.59  0.00   

   2019  2020   2021  2022  2023  2024  
0  2.60  2.93   2.78  2.76  3.03  0.00  
1  5.06  5.39   4.95  4.79  5.19  5.50  
2  9.26  9.77  10.06  9.65  9.44  8.65  
3  4.88  5.60   5.32  4.74  4.96  4.86  
4  0.00  0.00   0.00  0.00  0.00  0.00  


In [109]:
# Aplicar un gradiente de color (Heatmap) con Pandas
# Esto resalta visualmente dónde están los promedios más altos
df_estilo = (
    tabla_estadia.style
    .background_gradient(cmap='YlOrRd', axis=None) # Amarillo a Rojo
    .format(precision=2)
)

html_final = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Análisis de Estadía Hospitalaria</title>
    <link rel="stylesheet" type="text/css" href="https://cdn.datatables.net/1.13.6/css/jquery.dataTables.min.css">
    <style>
        body {{ font-family: 'Segoe UI', Arial, sans-serif; margin: 40px; background-color: #f4f7f6; }}
        .container {{ background: white; padding: 25px; border-radius: 12px; box-shadow: 0 4px 15px rgba(0,0,0,0.05); }}
        h2 {{ color: #1a5276; border-bottom: 2px solid #3498db; padding-bottom: 10px; }}
        .info-box {{ background: #e8f6f3; padding: 10px; border-left: 5px solid #1abc9c; margin-bottom: 20px; font-size: 0.9em; }}
        table.dataframe {{ width: 100% !important; border-collapse: collapse; }}
    </style>
</head>
<body>
    <div class="container">
        <h2>Promedio de Días de Estadía por Entidad</h2>
        <div class="info-box">
            <b>Nota:</b> Los colores indican la duración de la estadía. 
            <span style="color:#d35400">Rojo: Estadías más largas</span> | 
            <span style="color:#f1c40f">Amarillo: Estadías más cortas</span>.
        </div>
        
        {df_estilo.to_html(classes='display nowrap', table_id='tabla_estadia')}
        
    </div>

    <script src="https://code.jquery.com/jquery-3.7.0.js"></script>
    <script src="https://cdn.datatables.net/1.13.6/js/jquery.dataTables.min.js"></script>
    <script>
        $(document).ready(function() {{
            $('#tabla_estadia').DataTable({{
                "pageLength": 25,
                "scrollX": true,
                "language": {{ "search": "Buscar entidad:" }},
                "order": [[0, "asc"]]
            }});
        }});
    </script>
</body>
</html>
"""

with open("reporte_estadia_promedio.html", "w", encoding="utf-8") as f:
    f.write(html_final)

print("¡Hecho! Abre 'reporte_estadia_promedio.html' para ver la comparativa.")

¡Hecho! Abre 'reporte_estadia_promedio.html' para ver la comparativa.


### sidrome metabolico

In [110]:
# Agrupamos por entidad y año de egreso
dias_promedio_entidad = (
    base_general
    .filter(pl.col("sindrome_metabolico") == True) # Filtrar solo síndrome metabólico
    .with_columns(
        pl.col("dia_estad").cast(pl.Float64, strict=False),
        pl.col("anio_egr").cast(pl.Float64, strict=False)
    )
    .group_by(["entidad", "anio_egr"])
    .agg([
        pl.col("dia_estad").mean().alias("promedio_estadia")
    ])
    .to_pandas()
)

# Limpieza de años para evitar el error anterior
dias_promedio_entidad["anio_egr"] = pd.to_numeric(dias_promedio_entidad["anio_egr"]).astype(int)

# Creamos el pivot
tabla_estadia = dias_promedio_entidad.pivot_table(
    index="entidad",
    columns="anio_egr",
    values="promedio_estadia",
    aggfunc="first", # Ya es un promedio, así que 'first' o 'mean' funcionan igual
    fill_value=0
)

# Opcional: Redondear a 2 decimales para que no se vea feo el HTML
tabla_estadia = tabla_estadia.round(2)

# Limpiar nombres de ejes
tabla_estadia.columns.name = None
tabla_estadia.index.name = "Entidad Hospitalaria"
tabla_estadia = tabla_estadia.reset_index()

print(tabla_estadia.head())

                                Entidad Hospitalaria   2015   2016   2017  \
0                                    FISCOMISIONALES   0.00   0.00   0.00   
1          INSTITUTO ECUATORIANO DE SEGURIDAD SOCIAL   7.29   6.77   6.82   
2                    JUNTA BENEFICENCIA DE GUAYAQUIL  11.78  13.26  13.91   
3                     MINISTERIO DE DEFENSA NACIONAL   5.37   5.33   6.30   
4  MINISTERIO DE JUSTICIA, DERECHOS HUMANOS Y CULTOS   8.58  11.18   9.71   

    2018   2019   2020   2021   2022   2023  2024  
0   3.96   3.44   4.60   4.26   4.79   4.19  0.00  
1   6.94   7.18   6.59   6.31   6.45   6.76  7.33  
2  14.21  17.28  17.07  15.16  12.14  10.57  9.59  
3   6.04   6.16   7.16   6.11   5.93   5.70  5.71  
4   0.00   0.00   0.00   0.00   0.00   0.00  0.00  


In [ ]:
# Aplicar un gradiente de color (Heatmap) con Pandas
# Esto resalta visualmente dónde están los promedios más altos
df_estilo = (
    tabla_estadia.style
    .background_gradient(cmap='YlOrRd', axis=None) # Amarillo a Rojo
    .format(precision=2)
)

html_final = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Análisis de Estadía Hospitalaria de Sindrome Metabólico</title>
    <link rel="stylesheet" type="text/css" href="https://cdn.datatables.net/1.13.6/css/jquery.dataTables.min.css">
    <style>
        body {{ font-family: 'Segoe UI', Arial, sans-serif; margin: 40px; background-color: #f4f7f6; }}
        .container {{ background: white; padding: 25px; border-radius: 12px; box-shadow: 0 4px 15px rgba(0,0,0,0.05); }}
        h2 {{ color: #1a5276; border-bottom: 2px solid #3498db; padding-bottom: 10px; }}
        .info-box {{ background: #e8f6f3; padding: 10px; border-left: 5px solid #1abc9c; margin-bottom: 20px; font-size: 0.9em; }}
        table.dataframe {{ width: 100% !important; border-collapse: collapse; }}
    </style>
</head>
<body>
    <div class="container">
        <h2>Promedio de Días de Estadía por Entidad de Síndrome Metabólico<</h2>
        <div class="info-box">
            <b>Nota:</b> Los colores indican la duración de la estadía. 
            <span style="color:#d35400">Rojo: Estadías más largas</span> | 
            <span style="color:#f1c40f">Amarillo: Estadías más cortas</span>.
        </div>
        
        {df_estilo.to_html(classes='display nowrap', table_id='tabla_estadia')}
        
    </div>

    <script src="https://code.jquery.com/jquery-3.7.0.js"></script>
    <script src="https://cdn.datatables.net/1.13.6/js/jquery.dataTables.min.js"></script>
    <script>
        $(document).ready(function() {{
            $('#tabla_estadia').DataTable({{
                "pageLength": 25,
                "scrollX": true,
                "language": {{ "search": "Buscar entidad:" }},
                "order": [[0, "asc"]]
            }});
        }});
    </script>
</body>
</html>
"""

with open("reporte_estadia_promedio_sindrome.html", "w", encoding="utf-8") as f:
    f.write(html_final)

print("¡Hecho! Abre 'reporte_estadia_promedio_sindrome.html' para ver la comparativa.")

¡Hecho! Abre 'reporte_estadia_promedio_sindrome.html' para ver la comparativa.


## mortalidad
tasa de mortalidad= egresos hospitalarios por muerte/egresos hospitalarios totales.
- Esto es solo msp. 


- Estádisticas de hospital que más tasa de mortalidad tiene en el ecuador. dar ultimo año o hacer interactivo. top 10 hospital con mayor tasa de mortalidad por año

MAPA
- top 5 de capitulos con más mortalidad. 

- Mapa puntos criticos de gente que muere. A nivel de residencia. tamaño bolita donde hay más tasa de mortalidad.


In [116]:
# leer diccionario del archivo: notebooks/limpieza/nuevo_diccionario_cau221rx.txt
import ast

# 1. Leemos el contenido del archivo .txt
with open("../limpieza/nuevo_diccionario_cau221rx.txt", "r", encoding="utf-8") as f:
    contenido = f.read()

# 2. Limpiamos el texto para quedarnos solo con lo que está después del '='
# Esto elimina "dictionary = " y nos deja solo con el "{...}"
if "=" in contenido:
    str_diccionario = contenido.split("=", 1)[1].strip()
else:
    str_diccionario = contenido.strip()

# 3. Lo convertimos en un objeto diccionario real
nuevo_diccionario = ast.literal_eval(str_diccionario)

#msp_data=pd.read_csv("../data/egresos_msp_final.csv")
msp_data=pl.read_csv("../../data/data_hospitales/hospitales_msp_egresos_cie10.csv",schema_overrides={"area_res": pl.String,"cap221rx": pl.String})

msp_data=msp_data.with_columns(
    pl.col("cau221rx")
    .replace(nuevo_diccionario, default=None)
    .alias("cau221rx_std")
)
msp_data["cau221rx_std"].value_counts()


C:\Users\Kristian Mendoza\AppData\Local\Temp\ipykernel_25140\3682512578.py:23: DeprecationWarning:

the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)



cau221rx_std,count
str,u32
"""Traumatismos del tórax (S20-S2…",13357
"""Trastornos del sistema digesti…",3620
"""Personas con riesg pot.salud, …",1897
"""Trastornos de la glándula tiro…",4151
"""Trastornos mentales y del comp…",10701
…,…
"""Otras enfermedades degenerativ…",811
"""Ciertos trastornos que afectan…",467
"""Traumatismos del antebrazo y d…",44026


In [ ]:
# --- 1. Diccionario de Homologación Manual ---
# Esto resuelve los casos de zonas no delimitadas y errores de dedo específicos
correcciones_canton_dict = {
    ("ZONAS NO DELIMITADAS", "LAS GOLONDRINAS"): ("IMBABURA", "COTACACHI"),
    ("ZONAS NO DELIMITADAS", "EL PIEDRERO"): ("GUAYAS", "EL TRIUNFO"),
    ("ZONAS NO DELIMITADAS", "MANGA DEL CURA"): ("MANABI", "EL CARMEN"),
    ("LOJA", "OLEMDO"): ("LOJA", "OLMEDO"),
    ("CANAR","AZOQUES"): ("CANAR", "AZOGUES"),
    ("GUAYAS","EMPALME"): ("GUAYAS", "EL EMPALME"),
    ("GUAYAS","SALITRE (URBINA JADO)"):("GUAYAS","SALITRE"),
    ("SANTO DOMINGO DE LOS TSACHILAS","LA CONDORDIA"):("SANTO DOMINGO DE LOS TSACHILAS","LA CONCORDIA"),
    ("ZAMORA CHINCHIPE","YANTZAZA (YANZATZA)"):("ZAMORA CHINCHIPE","YANTZAZA"),
    ("IMBABURA","COTACAHI"): ("IMBABURA", "COTACACHI"),
    ("GUAYAS","CRNEL. MARCELINO MARIDUENA"):("GUAYAS","CORONEL MARCELINO MARIDUENA"),
    ("NAPO","CARLOS JULIO ARROSEMENA TOLA"):("NAPO","CARLOS JULIO AROSEMENA TOLA"),
    ("GUAYAS","GENERAL ANTONIO ELIZALDE (BUCAY)"):("GUAYAS","GENERAL ANTONIO ELIZALDE"),
    ("GUAYAS","GNRAL. ANTONIO ELIZALDE"):("GUAYAS","GENERAL ANTONIO ELIZALDE"),
    ("GUAYAS","ALFREDO BAQUERIZO MORENO (JUJAN)"):("GUAYAS","ALFREDO BAQUERIZO MORENO")
}

correcciones_parroquia_dict = {
    ("PICHINCHA","QUITO","QUITO"): ("PICHINCHA","QUITO","QUITO , CABECERA CANTONAL Y CAPITAL PROVINCIAL"),
    ("GUAYAS","GUAYAQUIL","GUAYAQUIL"): ("GUAYAS","GUAYAQUIL","GUAYAQUIL , CABECERA CANTONAL Y CAPITAL PROVINCIAL")}

def aplicar_correcciones_master_canton(df, sufijo):
    col_prov = f"prov_{sufijo}"
    col_cant = f"cant_{sufijo}"
    
    # 1. Limpieza inicial de strings
    df = df.with_columns([
        pl.col(col_prov).str.strip_chars().str.to_uppercase(),
        pl.col(col_cant).str.strip_chars().str.to_uppercase()
    ])
    
    # 2. Aplicar la lógica del diccionario
    for (p_old, c_old), (p_new, c_new) in correcciones_canton_dict.items():
        df = df.with_columns([
            pl.when((pl.col(col_prov) == p_old) & (pl.col(col_cant) == c_old))
            .then(pl.lit(p_new))
            .otherwise(pl.col(col_prov))
            .alias(col_prov),
            
            pl.when((pl.col(col_prov) == p_old) & (pl.col(col_cant) == c_old))
            .then(pl.lit(c_new))
            .otherwise(pl.col(col_cant))
            .alias(col_cant)
        ])
    return df

# Aplicar a la base original
msp_data = aplicar_correcciones_master_canton(msp_data, "res")
msp_data= aplicar_correcciones_master_canton(msp_data,  "ubi")



In [128]:
msp_data.columns

Index(['area_ubi', 'clase', 'tipo', 'entidad_x', 'sector', 'mes_inv',
       'nac_pac', 'nom_pais', 'cod_pais', 'sexo', 'cod_edad', 'edad', 'etnia',
       'prov_res', 'area_res', 'anio_ingr', 'mes_ingr', 'dia_ingr',
       'fecha_ingr', 'anio_egr', 'mes_egr', 'dia_egr', 'fecha_egr',
       'dia_estad', 'con_egrpa', 'esp_egrpa', 'cau_cie10', 'causa3',
       'cap221rx', 'cau221rx', 'cau298rx', 'cant_ubi_std_x', 'parr_ubi_std_x',
       'cant_res_std', 'parr_res_std', 'ruc', 'eod', 'unicodigo_geosalud',
       'nombre_centro_de_salud', 'nivel_de_atencion', 'direccion', 'longps',
       'latgps', 'code_provincia_x', 'provincia', 'canton', 'code_parroquia',
       'sri_parroquia', 'parroquia', 'cau221rx_std', 'lat_res', 'lon_res',
       'es_fallecido'],
      dtype='object')

In [125]:
print(msp_data["con_egrpa"].value_counts())

con_egrpa
Vivo                           4973775
Fallecido en 48 horas y más      79057
Fallecido menos de 48 horas      22781
Name: count, dtype: int64


In [136]:
#msp_data = msp_data.to_pandas()
inec_parroquias=pd.read_csv("../../data/parroquias_cantones_inec.csv")
inec_cantones=inec_parroquias[["provincia","canton","code_canton"]].drop_duplicates()
inec_cantones.rename(columns={"code_canton":"code_canton_res"},inplace=True)
msp_data=pd.merge(msp_data,inec_cantones[["provincia","canton","code_canton_res"]],left_on=["prov_res","cant_res_std"],right_on=["provincia","canton"],how="left")

In [137]:
msp_data.drop(columns=["canton_y","provincia_y"],inplace=True)
msp_data.rename(columns={"code_canton_y":"code_canton_res","code_canton_x":"code_canton_ubi","canton_y":"canton_res","provincia_y":"provincia_res"},inplace=True)

In [140]:
msp_data.columns

Index(['area_ubi', 'clase', 'tipo', 'entidad_x', 'sector', 'mes_inv',
       'nac_pac', 'nom_pais', 'cod_pais', 'sexo', 'cod_edad', 'edad', 'etnia',
       'prov_res', 'area_res', 'anio_ingr', 'mes_ingr', 'dia_ingr',
       'fecha_ingr', 'anio_egr', 'mes_egr', 'dia_egr', 'fecha_egr',
       'dia_estad', 'con_egrpa', 'esp_egrpa', 'cau_cie10', 'causa3',
       'cap221rx', 'cau221rx', 'cau298rx', 'cant_ubi_std_x', 'parr_ubi_std_x',
       'cant_res_std', 'parr_res_std', 'ruc', 'eod', 'unicodigo_geosalud',
       'nombre_centro_de_salud', 'nivel_de_atencion', 'direccion', 'longps',
       'latgps', 'code_provincia_x', 'provincia_x', 'canton_x',
       'code_parroquia', 'sri_parroquia', 'parroquia', 'cau221rx_std',
       'lat_res', 'lon_res', 'es_fallecido', 'code_canton_res'],
      dtype='object')

In [141]:

msp_data=msp_data.merge(
    cantones[["code_canton","lat_canton","lon_canton"]],
    left_on="code_canton_res",
    right_on="code_canton",
    how="left"
).drop(columns=["code_canton"])

In [142]:
msp_data["lat_canton"].isnull().sum()

5815

In [ ]:
import pandas as pd
#TODO: AÑADIR CANTON de ubicacion
# 1. Definir categorías de fallecidos
fallecidos_categorias = ["Fallecido en 48 horas y más", "Fallecido menos de 48 horas"]

# 2. Limpieza básica de años (por si vienen como 2015.0)
msp_data["anio_egr"] = pd.to_numeric(msp_data["anio_egr"], errors='coerce').fillna(0).astype(int)

# 3. Crear indicadores
# Marcamos con 1 si falleció, con 0 si no (Vivo)
msp_data['es_fallecido'] = msp_data['con_egrpa'].isin(fallecidos_categorias).astype(int)

# 4. Agrupar por hospital y año
mortalidad_stats = (
    msp_data.groupby(['nombre_centro_de_salud', 'anio_egr'])
    .agg(
        total_egresos=('con_egrpa', 'count'),
        total_fallecidos=('es_fallecido', 'sum')
    )
    .reset_index()
)

# 5. Calcular la tasa (Fallecidos / Total * 100)
mortalidad_stats['tasa_mortalidad'] = (
    mortalidad_stats['total_fallecidos'] / mortalidad_stats['total_egresos'] * 100
).round(2)

# 6. Filtrar para evitar ruido estadístico (mínimo 50 egresos)
# Y obtener el Top 10 por cada año
top_10_mortalidad = (
    mortalidad_stats[mortalidad_stats['total_egresos'] > 50]
    .sort_values(['anio_egr', 'tasa_mortalidad'], ascending=[True, False])
    .groupby('anio_egr')
    .head(10)
)

In [ ]:
#TODO: PERMITIR FILTRAR POR AÑO EN EL HTML (con DataTables o similar)

# Aplicar estilos con Pandas Styler
df_estilizado = (
    top_10_mortalidad.style
    .background_gradient(subset=['tasa_mortalidad'], cmap='Reds')
    .format({
        'tasa_mortalidad': '{:.2f}%', 
        'total_egresos': '{:,}', 
        'total_fallecidos': '{:,}'
    })
    .hide(axis="index")
)

html_template = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Reporte de Mortalidad Hospitalaria</title>
    <link rel="stylesheet" type="text/css" href="https://cdn.datatables.net/1.13.6/css/jquery.dataTables.min.css">
    <style>
        body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 40px; background-color: #f9f9f9; }}
        .container {{ background: white; padding: 30px; border-radius: 15px; box-shadow: 0 4px 20px rgba(0,0,0,0.08); }}
        h2 {{ color: #c0392b; border-bottom: 2px solid #e74c3c; padding-bottom: 10px; }}
        .alert {{ background-color: #fdf2f2; border-left: 6px solid #f05252; padding: 15px; margin-bottom: 20px; font-size: 0.9em; }}
        table.dataframe {{ width: 100% !important; border-collapse: collapse; margin-top: 20px; }}
    </style>
</head>
<body>
    <div class="container">
        <h2>Top 10 Hospitales con Mayor Tasa de Mortalidad por Año</h2>
        
        <div class="alert">
            <strong>Criterio Técnico:</strong> La tasa se calcula como <i>(Fallecidos / Total Egresos) * 100</i>. 
            Se excluyen centros con menos de 50 registros anuales para evitar sesgos por volumen bajo.
        </div>

        {df_estilizado.to_html(classes='display nowrap', table_id='tabla_mortalidad')}
        
    </div>

    <script src="https://code.jquery.com/jquery-3.7.0.js"></script>
    <script src="https://cdn.datatables.net/1.13.6/js/jquery.dataTables.min.js"></script>
    <script>
        $(document).ready(function() {{
            $('#tabla_mortalidad').DataTable({{
                "pageLength": 10,
                "order": [[ 1, "desc" ], [ 4, "desc" ]], // Ordena por Año (desc) y luego por Tasa (desc)
                "language": {{
                    "search": "Buscar hospital o año:",
                    "lengthMenu": "Mostrar _MENU_ registros",
                    "info": "Mostrando del _START_ al _END_ de _TOTAL_ hospitales"
                }}
            }});
        }});
    </script>
</body>
</html>
"""

# Guardar el archivo
with open("top_10_mortalidad_msp.html", "w", encoding="utf-8") as f:
    f.write(html_template)

print("¡Archivo 'top_10_mortalidad_msp.html' generado exitosamente!")

¡Archivo 'top_10_mortalidad_msp.html' generado exitosamente!


- top 5 de capitulos con más mortalidad. 


In [131]:
import pandas as pd

# 1. Definir categorías de fallecidos
fallecidos_categorias = ["Fallecido en 48 horas y más", "Fallecido menos de 48 horas"]

# 2. Asegurar que la columna de mortalidad sea numérica
msp_data['es_fallecido'] = msp_data['con_egrpa'].isin(fallecidos_categorias).astype(int)

# 3. Agrupar por Capítulo CIE-10 (cau221rx_std) y Año
# Nota: Si prefieres ver el acumulado de todos los años, quita 'anio_egr' del groupby
capitulos_mortalidad = (
    msp_data.groupby(['cau221rx_std', 'anio_egr'])
    .agg(
        total_egresos=('con_egrpa', 'count'),
        total_fallecidos=('es_fallecido', 'sum')
    )
    .reset_index()
)

# 4. Calcular la tasa de letalidad por capítulo
capitulos_mortalidad['tasa_mortalidad'] = (
    capitulos_mortalidad['total_fallecidos'] / capitulos_mortalidad['total_egresos'] * 100
).round(2)

# 5. Obtener el Top 5 por año (basado en número absoluto de fallecidos)
top_5_capitulos = (
    capitulos_mortalidad.sort_values(['anio_egr', 'total_fallecidos'], ascending=[True, False])
    .groupby('anio_egr')
    .head(5)
)

In [ ]:
#TODO: PERMITIR FILTRAR POR AÑO EN EL HTML (con DataTables o similar)
# Aplicar estilos
df_estilo_caps = (
    top_5_capitulos.style
    .background_gradient(subset=['total_fallecidos'], cmap='Oranges')
    .format({
        'tasa_mortalidad': '{:.2f}%',
        'total_egresos': '{:,}',
        'total_fallecidos': '{:,}'
    })
    .hide(axis="index")
)

html_caps = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Mortalidad por Capítulos CIE-10</title>
    <link rel="stylesheet" type="text/css" href="https://cdn.datatables.net/1.13.6/css/jquery.dataTables.min.css">
    <style>
        body {{ font-family: sans-serif; margin: 30px; background-color: #f4f4f9; }}
        .container {{ background: white; padding: 25px; border-radius: 10px; box-shadow: 0 4px 10px rgba(0,0,0,0.1); }}
        h2 {{ color: #2e4053; border-left: 5px solid #e67e22; padding-left: 15px; }}
        table {{ width: 100% !important; font-size: 0.95em; }}
    </style>
</head>
<body>
    <div class="container">
        <h2>Top 5 Capítulos CIE-10 con Mayor Número de Fallecidos</h2>
        <p>Este reporte muestra las causas principales de muerte hospitalaria agrupadas por los grandes capítulos de la Clasificación Internacional de Enfermedades.</p>
        
        {df_estilo_caps.to_html(classes='display', table_id='tabla_caps')}
        
    </div>
    <script src="https://code.jquery.com/jquery-3.7.0.js"></script>
    <script src="https://cdn.datatables.net/1.13.6/js/jquery.dataTables.min.js"></script>
    <script>
        $(document).ready(function() {{
            $('#tabla_caps').DataTable({{
                "order": [[ 1, "desc" ], [ 3, "desc" ]],
                "language": {{ "search": "Filtrar por año o capítulo:" }}
            }});
        }});
    </script>
</body>
</html>
"""

with open("mortalidad_por_capitulos.html", "w", encoding="utf-8") as f:
    f.write(html_caps)

print("Reporte generado: mortalidad_por_capitulos.html")

Reporte generado: mortalidad_por_capitulos.html


### Mapas

- Mapa puntos criticos de gente que muere. A nivel de residencia. tamaño bolita donde hay más tasa de mortalidad.

In [145]:
# 2. Filtrar y limpiar para el mapa
mapa_df = msp_data[[
    "anio_egr", "cau221rx_std", "lat_canton", "lon_canton", 
    "cant_res_std", "es_fallecido", "con_egrpa"
]].dropna(subset=["lat_canton", "lon_canton"])

# 3. Agrupación por ubicación de residencia y causa
# Agrupamos por coordenadas y nombre del cantón para no perder la ubicación del punto
puntos_criticos = (
    mapa_df.groupby(["cant_res_std", "lat_canton", "lon_canton", "cau221rx_std", "anio_egr"])
    .agg(
        total_egresos=("con_egrpa", "count"),
        total_fallecidos=("es_fallecido", "sum")
    )
    .reset_index()
)

# 4. Calcular Tasa de Mortalidad
puntos_criticos["tasa_mortalidad"] = (
    puntos_criticos["total_fallecidos"] / puntos_criticos["total_egresos"] * 100
).round(2)

# 5. Crear la categoría "TOTAL" (Agregado de todas las causas por ubicación)
total_por_ubicacion = (
    mapa_df.groupby(["cant_res_std", "lat_canton", "lon_canton", "anio_egr"])
    .agg(
        total_egresos=("con_egrpa", "count"),
        total_fallecidos=("es_fallecido", "sum")
    )
    .reset_index()
)
total_por_ubicacion["cau221rx_std"] = "--- TOTAL TODAS LAS CAUSAS ---"
total_por_ubicacion["tasa_mortalidad"] = (
    total_por_ubicacion["total_fallecidos"] / total_por_ubicacion["total_egresos"] * 100
).round(2)

# Unir ambos DataFrames
df_final_mapa = pd.concat([puntos_criticos, total_por_ubicacion], ignore_index=True)

In [146]:
mapa_mortalidad= KeplerGl(height=800)
mapa_mortalidad.add_data(data=df_final_mapa, name="mortalidad_ubicacion")
mapa_mortalidad

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


KeplerGl(data={'mortalidad_ubicacion': {'index': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17…

In [147]:
with open('config_mortalidad.py', 'w') as f:
   f.write('config_mortalidad = {}'.format(mapa_mortalidad.config))

In [ ]:
mapa_mortalidad.save_to_html(file_name="../../maps/for_presentation/processed/mortalidad_ubicacion.html")

Map saved to ../../maps/processed/mortalidad_ubicacion.html!


## Presupuesto
- gasto devengado en diciembre dividido al numero de egeresos y filtrado por clase ( especialdiad, o general, etc) Mapa.
- Gasto devengado en medicamentos y personal. a nivel Mapa. ajustado por egresos. 
- numeros de médicos y enfermeros. Mapa

In [22]:
lat_lon_hospitales=pd.read_csv("../../data/lat_lon_msp_hospitales.csv")
lat_lon_hospitales["latgps"]=pd.to_numeric(lat_lon_hospitales["latgps"], errors='coerce')
lat_lon_hospitales["longps"]=pd.to_numeric(lat_lon_hospitales["longps"], errors='coerce')

In [23]:
df_finanzas=pd.read_csv("../../data/data_hospitales/df_finanzas_dec2023.csv")
df_recursos_humanos=pd.read_csv("../../data/data_hospitales/df_recursos_humanos_dec2023.csv")

df_recursos_humanos.rename(columns={"health_center_name":"nombre_centro_de_salud"},inplace=True)
df_finanzas.rename(columns={"health_center_name":"nombre_centro_de_salud"},inplace=True)
# Unir ambos DataFrames por nombre del centro de salud
df_recursos_humanos = pd.merge(df_recursos_humanos, lat_lon_hospitales, on="nombre_centro_de_salud", how="left")
df_finanzas = pd.merge(df_finanzas, lat_lon_hospitales, on="nombre_centro_de_salud", how="left")


display(df_finanzas[df_finanzas["latgps"].isnull()]["nombre_centro_de_salud"])


Series([], Name: nombre_centro_de_salud, dtype: object)

In [24]:
mapa_gasto_egresos=KeplerGl(height=800)
mapa_gasto_egresos.add_data(data=df_finanzas, name="finanzas")
mapa_gasto_egresos

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


c:\Users\Kristian Mendoza\.conda\envs\rodri_msp_geo\Lib\site-packages\jupyter_client\session.py:721: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)


KeplerGl(data={'finanzas': {'index': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20…

In [ ]:
with open('config_gasto_egresos.py', 'w') as f:
    f.write('config_gasto_egresos = {}'.format(mapa_gasto_egresos.config))
mapa_gasto_egresos.save_to_html(file_name="../../maps/for_presentation/processed/gasto_egresos.html")

Map saved to ../../maps/processed/gasto_egresos.html!


In [26]:
mapa_recursos_humanos=KeplerGl(height=800)
mapa_recursos_humanos.add_data(data=df_recursos_humanos, name="recursos_humanos")
mapa_recursos_humanos

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


KeplerGl(data={'recursos_humanos': {'index': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18…

In [ ]:
with open('config_recursos_humanos.py', 'w') as f:
    f.write('config_recursos_humanos = {}'.format(mapa_recursos_humanos.config))
    
mapa_recursos_humanos.save_to_html(file_name="../../maps/for_presentation/processed/recursos_humanos.html")

Map saved to ../../maps/processed/recursos_humanos.html!


## para presentar

Mapas sacar los html, super bien agrupados. para poder incrustarlos en un html con formato como la presentación de pptx. Utilizar los colores del hub